# Gold Layer - Cross-Category Purchase Patterns View

## Purpose
Identify which departments are frequently purchased together in the same order for cross-category merchandising and bundling strategies.

## Type
**SQL View** (lightweight, real-time)

## Input
* **Source:** `big_data.silver.order_products` (33.8M rows)
* **Source:** `big_data.silver.products_enriched` (49.7K rows)

## Output
* **Target:** `big_data.gold.vw_cross_category_patterns`
* **Rows:** ~18 department pairs
* **Refresh:** Real-time (always reflects current Silver data)

## Use Cases
* 🏪 Cross-category merchandising
* 📦 Bundle creation (e.g., "Produce + Dairy" combo)
* 📈 Category affinity analysis

## Why View (not Table)?
* ✅ Result is very small (~18 rows)
* ✅ Query is fast (simple join + groupBy)
* ✅ Always synchronized with Silver
* ✅ Zero storage overhead

## SQL Logic
1. Join order_products with products to get department
2. Self-join on order_id where departments differ
3. Get distinct department pairs
4. COUNT orders per department pair

## Execution
Run all cells sequentially. Expected runtime: ~30 seconds.

In [0]:
%sql
-- Cross-Category Purchase Patterns View

CREATE OR REPLACE VIEW big_data.gold.vw_cross_category_patterns AS
WITH enriched AS (
  SELECT 
    op.order_id,
    op.product_id,
    p.department
  FROM big_data.silver.order_products op
  LEFT JOIN big_data.silver.products_enriched p ON op.product_id = p.product_id
),
dept_pairs AS (
  SELECT DISTINCT
    a.department AS dept1,
    b.department AS dept2
  FROM enriched a
  JOIN enriched b ON a.order_id = b.order_id
  WHERE a.department != b.department
)
SELECT 
  dept1,
  dept2,
  COUNT(*) AS order_count
FROM dept_pairs
GROUP BY dept1, dept2
ORDER BY order_count DESC;

In [0]:
%sql
-- Verify view exists and preview top department pairs
SELECT * FROM big_data.gold.vw_cross_category_patterns
LIMIT 10;